# 12: demo na prawdziwych zdjęciach (wiatrak) z modelem z notebooka 10

Bierze dwie klatki z Twoich telefonów (A i B, B obrócony o 90°), ustawia je tak, jak oczekuje model, i pokazuje
wynik **najnowszego modelu z U-Netem** (`*_raft_unet_ft_mix` z `10_mix_layers_2`).

**Co zrobić:**
1. Skopiuj dwie klatki na Drive do `MyDrive/rs_project/real/wiatrak/` i nazwij je `kamera_A.png` (albo `.jpg`) i `kamera_B.png`.
   Zostaw je **tak, jak są** (nie obracaj ręcznie), notebook sam je ustawi.
2. Uruchom komórki po kolei: SETUP, 1, 2, 3.
3. Komórka 2 sama dobiera obroty tak, żeby oba obrazy pokazywały tę samą scenę i pasy rolling shutter szły zgodnie z założeniem modelu.
   Ponieważ nie wiemy, w którą stronę (np. z góry na dół czy z dołu do góry) telefon czyta matrycę, komórka 3 pokazuje
   **4 możliwości** (4 wiersze). Wybierz ten wiersz, w którym wiatrak wygląda najbardziej naturalnie.

To ocena na oko: nie ma klatki bez rolling shutter do porównania.

In [ ]:
# === SETUP: uruchamiaj zawsze jako pierwszą komórkę ===
REPO = 'kordiusz/RollingShutter'
BRANCH = 'filip-layers'

import os, sys, subprocess, getpass, importlib
from google.colab import drive
drive.mount('/content/drive')

TOKEN_FILE = '/content/drive/MyDrive/rs_project/.github_token'   # opcjonalnie: token zapisany na Drive
CODE = '/content/code'

def _git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(token, '***') if 'token' in globals() else r.stderr
        raise RuntimeError('git ' + args[0] + ' nie powiodło się:\n' + msg)
    return r.stdout

if not os.path.exists(CODE):
    token = (open(TOKEN_FILE).read().strip() if os.path.exists(TOKEN_FILE)
             else getpass.getpass('GitHub token (fine-grained, tylko odczyt): '))
    _git('clone', '-q', '-b', BRANCH, f'https://{token}@github.com/{REPO}.git', CODE)
else:
    _git('-C', CODE, 'fetch', '-q', 'origin', BRANCH)
    _git('-C', CODE, 'checkout', '-q', '-B', BRANCH, 'FETCH_HEAD')
os.chdir(CODE)
sys.path.insert(0, f'{CODE}/src')
import project_paths as P
importlib.reload(P)
P.ensure_dirs()
print(_git('log', '-1', '--format=kod: %h  %s  (%cd)  gałąź ' + BRANCH, '--date=short'))

## 1. Ustawienia
Co ustawiasz sam:
- `T_FULL_MS`: czas odczytu całej klatki (zmierzony diodą: 5,49 ms).
- `STRIPES_A`, `STRIPES_B`: jak wyglądają pasy rolling shutter w Twoich plikach, **bez żadnego obracania**
  (`'vertical'` = pionowe, `'horizontal'` = poziome). U Ciebie obie klatki mają pionowe.
- `SYNC_SWEEP`: o ile ms mógł się różnić start odczytu telefonów (przy 120 fps rozsądnie: od -4 do 4).
- `SIDE`: rozmiar wycinka w pikselach oryginału (maks. 1080). `CENTER_A`: środek wycinka (x, y) w obrazie A po obróceniu,
  `None` = środek.

In [ ]:
import cv2, numpy as np, torch
import demo_real
importlib.reload(demo_real)
from demo_real import prepare_pair, estimate_static_offset, to_bgr8
from raft_unet import load_model
from IPython.display import Image, display

REAL = P.ROOT / 'real' / 'wiatrak'
FILE_A = sorted(REAL.glob('kamera_A.*'))[0]
FILE_B = sorted(REAL.glob('kamera_B.*'))[0]

T_FULL_MS = 5.49
STRIPES_A = 'vertical'
STRIPES_B = 'vertical'
SYNC_SWEEP = [-4, -2, 0, 2, 4]
SIDE = 1024
CENTER_A = None

RUN = P.find_run('raft_unet_ft_mix') or P.find_run('raft_unet_mix') or P.find_run('raft_unet_ft')
print('model:', RUN)
assert RUN, 'Brak wytrenowanego modelu (runs/*_raft_unet_ft_mix)'
img_a0, img_b0 = cv2.imread(str(FILE_A)), cv2.imread(str(FILE_B))
print('A', FILE_A.name, img_a0.shape, '| B', FILE_B.name, img_b0.shape)

## 2. Ustawienie obrazów
Model zakłada, że kamera A czyta **wiersze** (pasy poziome), a kamera B **kolumny** (pasy pionowe, po obróceniu B do pozycji A).
Twoje pliki mają pasy pionowe, więc A trzeba obrócić o 90°, a B tak, żeby pokazywał tę samą scenę co A.
Dopasowanie sceny sprawdzam automatycznie (wspólne punkty charakterystyczne ORB: im więcej, tym lepsze dopasowanie).
Wartości `ka` / `kb` to liczba obrotów o 90° (`np.rot90`) zastosowanych do A i B.

In [ ]:
def rot(img, k):
    return np.ascontiguousarray(np.rot90(img, k))

def match_score(a, b, width=800):
    """Liczba zgodnych punktów (ORB + RANSAC) między dwoma obrazami: im więcej, tym bardziej ta sama scena."""
    def prep(im):
        s = width / max(im.shape[:2])
        return cv2.cvtColor(cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2GRAY)
    ga, gb = prep(a), prep(b)
    orb = cv2.ORB_create(4000)
    ka, da = orb.detectAndCompute(ga, None)
    kb, db = orb.detectAndCompute(gb, None)
    if da is None or db is None:
        return 0
    m = cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(da, db, k=2)
    good = [p[0] for p in m if len(p) == 2 and p[0].distance < 0.75 * p[1].distance]
    if len(good) < 8:
        return len(good)
    src = np.float32([ka[g.queryIdx].pt for g in good]).reshape(-1, 1, 2)
    dst = np.float32([kb[g.trainIdx].pt for g in good]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    return int(mask.sum()) if mask is not None else 0

# A po obrocie ma mieć pasy poziome, B po ustawieniu do pozycji A: pionowe
KA_OPTS = (1, 3) if STRIPES_A == 'vertical' else (0, 2)
KB_OPTS = (0, 2) if STRIPES_B == 'vertical' else (1, 3)

cands = []
for ka in KA_OPTS:
    a = rot(img_a0, ka)
    sc = {kb: match_score(a, rot(img_b0, kb)) for kb in KB_OPTS}
    kb = max(sc, key=sc.get)
    print(f'ka={ka}: dopasowanie B dla kb={dict(sc)} -> wybieram kb={kb}')
    for rk in (1, 3):       # rk: kierunek odczytu kamery B (1 albo 3 = -1)
        cands.append(dict(ka=ka, kb=kb, rot_k=rk, score=sc[kb]))
if max(c['score'] for c in cands) < 15:
    print('UWAGA: bardzo mało wspólnych punktów, sprawdź, czy A i B widzą tę samą scenę (i czy STRIPES_* są dobrze ustawione).')

## 3. Wynik (4 możliwości)
Każdy wiersz to inne założenie o kierunku odczytu matrycy (`ka`: kierunek w A, `rot_k`: kierunek w B).
W wierszu: pierwsza kratka = wejście z kamery A, kolejne = wynik modelu dla różnych przesunięć startu odczytu (`sync`).
Obrazy są obrócone z powrotem do pozycji z Twoich plików. Wybierz wiersz i kratkę, gdzie łopaty wyglądają najprościej.

In [ ]:
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_grad_enabled(False)
model, ck = load_model(str(RUN), dev)
assert ck['mode'] == 'refine', f"oczekiwano modelu refine, jest {ck['mode']}"
f = lambda v: torch.tensor([float(v)], device=dev)

def lab(im, t):
    im = im.copy()
    cv2.rectangle(im, (0, 0), (im.shape[1], 18), (255, 255, 255), -1)
    cv2.putText(im, t, (3, 13), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (0, 0, 0), 1, cv2.LINE_AA)
    return im

OUT = P.RESULTS / 'demo_wiatrak_unet'; OUT.mkdir(parents=True, exist_ok=True)
rows = []
for i, c in enumerate(cands):
    a_proc, b_back = rot(img_a0, c['ka']), rot(img_b0, c['kb'])
    model.geom.rot_k = c['rot_k']
    pair = prepare_pair(a_proc, rot(b_back, c['rot_k']), c['rot_k'], CENTER_A, SIDE, 256, T_FULL_MS)
    a, b = pair['rs_a'].to(dev), pair['rs_b'].to(dev)
    d = model.geom.flow_ab(a, b)
    bv = estimate_static_offset(d)
    back = lambda im: rot(im, -c['ka'])          # z powrotem do pozycji z pliku A
    tiles = [lab(back(pair['crop_a']), f'{i}: wejscie A (ka={c["ka"]}, rot_k={c["rot_k"]})')]
    for s in SYNC_SWEEP:
        pred, _ = model(a, b, f(pair['T']), f(pair['sync'] + s), bv)
        tiles.append(lab(back(to_bgr8(pred)), f'{i}: sync {s:+g} ms'))
    rows.append(np.hstack(tiles))
    print(f"{i}: ka={c['ka']} kb={c['kb']} rot_k={c['rot_k']} | T wycinka {pair['T']:.2f} ms | "
          f"sync srodkow {pair['sync']:+.2f} ms | przesuniecie widokow {np.round(pair['shift'], 1)} px")
img = np.vstack(rows)
cv2.imwrite(str(OUT / 'wynik.png'), img)
display(Image(str(OUT / 'wynik.png')))
print('zapisano', OUT / 'wynik.png')

## 4. Jak czytać wynik
- Jeśli we wszystkich wierszach wynik wygląda jak wejście, to zniekształcenia przy odczycie 5,49 ms są po prostu małe
  (albo model nie umie ich naprawić). To też jest wynik.
- Wiersz, w którym po korekcji łopaty są wyraźnie prostsze niż na wejściu, wskazuje właściwy kierunek odczytu. Zapisz jego `ka` i `rot_k`.
- Potem zawęź `SYNC_SWEEP` wokół najlepszego `sync` (np. `[-1, -0.5, 0, 0.5, 1]`) i uruchom komórkę 3 ponownie.
- Przy 5,49 ms model jest poza zakresem, na którym był trenowany (odczyty 15 do 33 ms), więc słaby wynik nie musi oznaczać błędu w danych.